# Session 3 — Probability Distributions

**Goal:** put a *shape* on each input, not just a mean and a variance — decide which
standard distribution models a column, check that choice instead of assuming it, and
know what to do when the check fails.

## What this stage does for the system

Session 2 left an open question: `chol` has skew `+1.11`, so `mean ± std` describes a
symmetric spread that column does not have. A distribution is the fix. Choosing one
is not a cosmetic act — it is an assumption that gets *used*:

- Session 4's confidence intervals assume the **sampling distribution** is Normal.
- Session 6 and 7 compute p-values from a **t-distribution**, valid only under
  approximate Normality (or a large enough sample).
- Session 8's chi-square test has its own distributional requirement on cell counts.
- Session 9's regression diagnostics check whether the **residuals** are Normal.

Every one of those inherits whatever you assume here. An unchecked Normality
assumption on a skewed column does not raise an error anywhere downstream; it just
returns a p-value that is quietly wrong. So the deliverable of this session is not
"`chol` is log-normal" — it is the *habit of checking*, and the two escape routes
(transform the variable, or switch to a method that does not need the assumption)
when the check fails.

## The dataset

Every session in this module works on one registry: the UCI **Heart Disease**
dataset (Cleveland), fetched live from the UCI ML Repository with `ucimlrepo` so the
notebooks are runnable by anyone without a CSV sitting on their machine. It holds 303
patients with clinical measurements (`age`, `trestbps` resting blood pressure, `chol`
serum cholesterol, `thalach` max heart rate achieved, `oldpeak` ST depression),
categorical findings (`sex`, `cp` chest-pain type, `fbs` fasting blood sugar > 120,
`restecg`, `exang` exercise-induced angina, `slope`, `ca`, `thal`), and the outcome
`num` — angiographic disease severity 0-4, which this module binarises into
`target` (0 = no disease, 1 = disease present).

Deliberately one dataset throughout: switching datasets between topics would mean
re-learning the data every session instead of building cumulative familiarity with
one problem, the way a real analyst does.

## How to read this notebook

Every code cell is followed by a short **Observe / Infer** note: *Observe* points at
exactly what to look at in that cell's output, and *Infer* explains what conclusion to
draw from it — and what a different result would imply. Read them before running the
next cell; several of them flag things worth double-checking before you move on.

## Prerequisites

This session runs entirely locally — no account or credentials needed.

```bash
pip install ucimlrepo pandas numpy scipy scikit-learn statsmodels matplotlib seaborn
```

## Step 1 — Load the registry from the UCI repository

Fetching directly from the UCI ML Repository keeps this notebook runnable by anyone,
instead of depending on a CSV already sitting on your machine. The same nine lines
open every session in this module, so the 297 patients below are the identical 297
patients every other notebook analyses.

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

heart_disease = fetch_ucirepo(id=45)
df = pd.concat([heart_disease.data.features, heart_disease.data.targets], axis=1)

# `num` is severity 0-4; this module screens for disease presence, so binarise it.
df = df.dropna().reset_index(drop=True)
df["target"] = (df["num"] > 0).astype(int)
df = df.drop(columns="num")

print(f"{len(df)} patients, {len(df.columns)} columns")
print(f"disease prevalence: {df['target'].mean():.3f}")
df.head()

**Observe:** `297 patients, 14 columns` and `disease prevalence: 0.461`. The preview
shows `age`, `sex`, `cp`, `trestbps`, `chol`, `fbs`, `restecg`, `thalach`, `exang`,
`oldpeak`, `slope`, `ca`, `thal`, and the `target` column just derived.
**Infer:** 303 rows are fetched and 297 survive `dropna()` — six patients are missing
`ca` (number of major vessels seen on fluoroscopy) or `thal`. Dropping six rows out of
303 is defensible here and keeps every notebook in this module working on the identical
297 patients; on a larger fraction of missing values you would have to impute instead,
and *that* choice would itself need the distribution work of Session 3. If your row
count is not 297, you are on a different subset than every number quoted below.

## Step 2 — Fit a Normal to a well-behaved input

`thalach` (maximum heart rate achieved during a stress test) had the mildest skew of
the continuous columns in Session 2, so it is the natural candidate for a Normal
model. Fitting one means nothing more than estimating $\mu$ and $\sigma$ — but
overlaying the fitted curve on the histogram is what turns "I estimated two numbers"
into "I checked whether those two numbers describe the data".

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

mu, sigma = df["thalach"].mean(), df["thalach"].std()
print(f"thalach: mu={mu:.1f}, sigma={sigma:.1f}, skew={stats.skew(df['thalach']):+.2f}")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(df["thalach"], bins=25, density=True, alpha=0.6,
        color="steelblue", edgecolor="white", label="registry")
xs = np.linspace(df["thalach"].min(), df["thalach"].max(), 300)
ax.plot(xs, stats.norm.pdf(xs, mu, sigma), "r-", lw=2,
        label=f"Normal({mu:.0f}, {sigma:.0f})")
ax.set_xlabel("thalach (max heart rate, bpm)")
ax.set_ylabel("density")
ax.set_title("Does a Normal describe max heart rate?")
ax.legend()
plt.tight_layout()
plt.show()

**Observe:** `mu=149.6, sigma=22.9, skew=-0.53`. The red curve tracks the histogram
closely through the middle, but the data's left tail extends further below the curve
than the right tail extends above it.
**Infer:** this is a good-enough Normal fit for the uses listed at the top of the
notebook — good enough is the right standard, since no real measurement is exactly
Normal. The mild left tail is not noise: maximum heart rate has a hard physiological
ceiling near 200 that compresses the right side, while illness and beta-blockers can
push the left side arbitrarily low. Worth remembering when Session 7 runs a t-test on
this exact column; the t-test is robust to skew this mild at n=297, and Session 7
checks that claim rather than asserting it.

## Step 3 — The empirical rule, as a numeric check

If a column really is Normal, 68.3% / 95.4% / 99.7% of values should fall within 1, 2,
and 3 standard deviations of the mean. Comparing the actual fractions against those
targets is a quicker, less visual check than a Q-Q plot, and it says *where* the fit
breaks rather than just whether it does.

In [ ]:
print(f"{'k':>2} {'observed':>9} {'Normal':>9} {'gap':>8}")
for k in [1, 2, 3]:
    observed = ((df["thalach"] >= mu - k*sigma) & (df["thalach"] <= mu + k*sigma)).mean()
    theoretical = stats.norm.cdf(k) - stats.norm.cdf(-k)
    print(f"{k:>2} {observed:9.3f} {theoretical:9.3f} {observed - theoretical:+8.3f}")

**Observe:** `0.663` vs `0.683` at k=1, `0.963` vs `0.954` at k=2, and an exact match
at k=3 — gaps of two percentage points at most.
**Infer:** slightly *fewer* points than expected within 1 sd and slightly *more* within
2 sd is the signature of a distribution marginally flatter in the middle than a
Normal, which is a minor deviation and not one that changes any decision downstream.
Sanity-check the direction of any bigger gap you see: far too few points inside 2 sd
means heavy tails (outliers dominating), far too many means the spread is being
overstated, usually by a handful of extreme values inflating `sigma`. Note this check
is symmetric by construction, so it cannot detect skew on its own — it would report a
lopsided distribution as a fine fit. That is what the next step is for.

## Step 4 — When Normal is the wrong model: `chol`

Session 2 flagged `chol` at skew `+1.11`. Two checks that catch what Step 3 cannot: a
**Q-Q plot**, which plots the data's quantiles against a Normal's and bends away from
the diagonal wherever the shapes diverge, and the **Shapiro-Wilk test**, which returns
a p-value for the null hypothesis "this sample came from a Normal distribution".

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].hist(df["chol"], bins=30, density=True, alpha=0.6,
             color="indianred", edgecolor="white")
c_mu, c_sigma = df["chol"].mean(), df["chol"].std()
xs = np.linspace(df["chol"].min(), df["chol"].max(), 300)
axes[0].plot(xs, stats.norm.pdf(xs, c_mu, c_sigma), "k-", lw=2, label="fitted Normal")
axes[0].set_xlabel("chol (mg/dL)"); axes[0].set_ylabel("density")
axes[0].set_title(f"chol, skew = {stats.skew(df['chol']):+.2f}")
axes[0].legend()

stats.probplot(df["chol"], dist="norm", plot=axes[1])
axes[1].set_title("Q-Q plot: chol vs Normal")
plt.tight_layout()
plt.show()

for col in ["thalach", "chol"]:
    result = stats.shapiro(df[col])
    verdict = "reject Normality" if result.pvalue < 0.05 else "cannot reject"
    print(f"Shapiro-Wilk {col:8} p={result.pvalue:.2e}  -> {verdict}")

**Observe:** the histogram is piled to the left with a thin tail stretching to 564; the
Q-Q plot's points curve upward away from the line at the right end; and Shapiro-Wilk
rejects Normality for `chol` (`p ≈ 1e-08`) — *and also for `thalach`* (`p ≈ 9e-05`).
**Infer:** the `thalach` rejection is the more instructive result. Step 2 showed
`thalach` is a perfectly usable Normal fit for practical purposes, yet the test rejects
it, because at n=297 Shapiro-Wilk has enough power to detect deviations too small to
matter. This is a general property of Normality tests: on small samples they fail to
reject anything, and on large samples they reject everything, which makes the p-value
alone a poor basis for deciding. Use the plot to judge *how badly* the assumption is
violated and the test only as corroboration. By that standard `thalach` is fine and
`chol` is not — the Q-Q divergence for `chol` is visible from across the room.

## Step 5 — Escape route 1: transform the variable

A right-skewed, strictly positive quantity is often **log-normal** — meaning
$\log(X)$ is Normal even though $X$ is not. If so, taking logs buys back every
Normality-assuming tool downstream, at the cost of working in log units.

In [ ]:
log_chol = np.log(df["chol"])

print(f"chol      skew={stats.skew(df['chol']):+.2f}  Shapiro p={stats.shapiro(df['chol']).pvalue:.2e}")
print(f"log(chol) skew={stats.skew(log_chol):+.2f}  Shapiro p={stats.shapiro(log_chol).pvalue:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(df["chol"], bins=30, color="indianred", edgecolor="white")
axes[0].set_title("chol (raw)"); axes[0].set_xlabel("mg/dL")
axes[1].hist(log_chol, bins=30, color="seagreen", edgecolor="white")
axes[1].set_title("log(chol)"); axes[1].set_xlabel("log mg/dL")
plt.tight_layout()
plt.show()

**Observe:** skew collapses from `+1.11` to `+0.06`, and Shapiro-Wilk goes from
`p ≈ 1e-08` (reject) to `p = 0.109` (cannot reject) — the same test that emphatically
rejected the raw column now finds no evidence against Normality for the logged one.
**Infer:** `chol` is close to log-normal, which is exactly what you would expect from a
quantity produced by many small multiplicative effects rather than additive ones.
Getting a "cannot reject" from a test that just rejected the untransformed column, on
the same 297 patients, is about as clean a demonstration as this dataset offers. Two
practical costs. A coefficient on `log(chol)` in Session 9's regression means "effect
of a 1% change in cholesterol", not "of 1 mg/dL", so every interpretation has to be
restated. And the transform only works on strictly positive columns — `oldpeak`, the
other strongly skewed column, contains genuine zeros, so `log` is undefined there and
you would need `log1p` or a square root instead.

## Step 6 — Discrete distributions: Bernoulli and Binomial

The outcome itself is discrete. One patient's `target` is a **Bernoulli(p)** trial.
The count of diseased patients among the next $n$ arrivals — assuming they are
independent and share the same $p$ — is **Binomial(n, p)**.

In [ ]:
p_disease = df["target"].mean()
n_next = 20

outcomes = np.arange(0, n_next + 1)
probs = stats.binom.pmf(outcomes, n_next, p_disease)

print(f"Bernoulli p = {p_disease:.3f}")
print(f"Binomial(n={n_next}, p={p_disease:.3f}): mean={n_next * p_disease:.1f}, "
      f"sd={np.sqrt(n_next * p_disease * (1 - p_disease)):.2f}")
print(f"P(15 or more of the next 20 have disease) = {1 - stats.binom.cdf(14, n_next, p_disease):.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(outcomes, probs, color="steelblue", edgecolor="white")
ax.axvline(n_next * p_disease, color="red", ls="--", label=f"mean = {n_next * p_disease:.1f}")
ax.set_xlabel("diseased patients among the next 20")
ax.set_ylabel("probability")
ax.set_title("Binomial distribution of the next clinic day")
ax.legend()
plt.tight_layout()
plt.show()

**Observe:** mean `9.2`, sd `2.23`, and `P(15 or more) = 0.0085` — under 1%.
**Infer:** that last number is a usable operational threshold: a day with 15+ positives
out of 20 would happen roughly once in 120 clinic days by chance alone, so seeing two
in a month is a reason to ask what changed in the referral stream rather than to shrug.
That is the seed of the drift monitoring in the MLOps module. But check the
assumptions before leaning on it: Binomial requires independent patients with a
*constant* p, and both can fail here — a screening campaign changes who gets referred
(p moves), and family members referred together are not independent. Either violation
widens the true spread beyond the `2.23` printed above, making a surprising day less
surprising than this model claims.

## Step 7 — Poisson: counts of a rare event over an interval

Where Binomial counts successes in a *fixed number of trials*, **Poisson(λ)** counts
events in a *fixed interval* when there is no natural upper bound — referrals per
month, alerts per day, admissions per shift. It is the right tool for capacity
planning, which is a question no accuracy metric can answer.

In [ ]:
n_disease_patients = int(df["target"].sum())
referrals_per_patient_per_year = 0.3          # illustrative operational rate
lam = n_disease_patients * referrals_per_patient_per_year / 12   # per month

print(f"{n_disease_patients} diseased patients, {referrals_per_patient_per_year}/yr each")
print(f"lambda = {lam:.2f} specialist referrals per month")
print(f"P(more than 4 in a month) = {1 - stats.poisson.cdf(4, lam):.3f}")
print(f"P(zero in a month)        = {stats.poisson.pmf(0, lam):.3f}")

counts = np.arange(0, 13)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(counts, stats.poisson.pmf(counts, lam), color="darkorange", edgecolor="white")
ax.axvline(lam, color="black", ls="--", label=f"lambda = {lam:.2f}")
ax.set_xlabel("referrals in a month"); ax.set_ylabel("probability")
ax.set_title("Poisson model for monthly referral load")
ax.legend()
plt.tight_layout()
plt.show()

**Observe:** `lambda = 3.43` per month, with `P(more than 4) = 0.260` and
`P(zero) = 0.033`.
**Infer:** staffing for the mean of 3.43 leaves you short about one month in four —
which is the entire reason to model the distribution rather than quote the average.
Note the defining property on display: for a Poisson, the variance *equals* the mean,
so spread is not a free parameter you can tune. Real referral counts are usually
**overdispersed** (variance above the mean) because referrals cluster — one clinic day
generates several — and a Poisson fitted to clustered data understates the busy months
specifically. If you ever fit this to real counts and the sample variance clearly
exceeds the sample mean, that is the signal to move to a negative binomial.

## Step 8 — Choosing a distribution, as a decision rule

Collecting the reasoning above into something reusable: the choice follows from what
kind of quantity you have, and the check follows from the choice.

In [ ]:
guide = pd.DataFrame([
    ("binary outcome, one patient",        "Bernoulli(p)",   "target",       "p is stable across patients"),
    ("count of successes in n patients",   "Binomial(n, p)", "target sum",   "independent, constant p"),
    ("count of events per time interval",  "Poisson(lambda)","referrals",    "variance == mean"),
    ("continuous, roughly symmetric",      "Normal(mu, sd)", "thalach, age", "Q-Q plot near diagonal"),
    ("continuous, positive, right-skewed", "Log-normal",     "chol",         "log(X) passes the Normal check"),
    ("continuous, skewed with zeros",      "no clean fit",   "oldpeak",      "use rank-based methods instead"),
], columns=["what you have", "distribution", "example here", "assumption to check"])
print(guide.to_string(index=False))

**Observe:** six rows, each pairing a distribution with the assumption that has to hold
for it — and one row (`oldpeak`) where the honest answer is that no standard
distribution fits well.
**Infer:** the last row is the one to take seriously. `oldpeak` is a mass of zeros plus
a right-skewed continuum, which no single named distribution captures, and forcing one
on it would be worse than admitting the gap. That admission has a concrete downstream
consequence: when Session 7 needs to compare `oldpeak` between the two outcome groups,
the parametric t-test is not the right tool and the rank-based Mann-Whitney U is —
which is exactly escape route 2, and the reason Session 7 spends a step on it.

## What this session hands to the next one

- **A per-column distributional verdict**: `thalach` and `age` Normal-enough,
  `chol` log-normal, `oldpeak` neither.
- **The checking habit** — Q-Q plot first, Normality test as corroboration, never the
  p-value alone — reused everywhere an assumption is made from here on.
- **Two escape routes** when the check fails: transform the column (Step 5), or use a
  method that does not need the assumption (Session 7's Mann-Whitney U).
- **The Normal distribution itself**, which Session 4 needs immediately: the Central
  Limit Theorem says the *sampling distribution of a mean* is Normal even when the
  underlying column is not, and that is what makes confidence intervals possible on a
  skewed column like `chol`.

## Try it yourself

1. Run the Step 3 empirical-rule check on `chol` instead of `thalach`. Which of the
   three k values goes wrong first, and what does that say about where the fit breaks?
2. Try `np.sqrt` and `np.log1p` on `oldpeak`. Does either get the skew below 0.5? What
   does the histogram look like afterwards, and does the zero mass survive?
3. In Step 6, recompute `P(15 or more of 20)` using the disease rate among patients
   over 55 instead of the overall rate. How much does the "surprising day" threshold
   move when the referral mix changes?
4. Fit a Normal to `thalach` *within each disease group* separately. Are the two
   group-specific fits better than the single pooled fit — and what would that imply
   about modelling the pooled column at all?